# 8. Data pipeline

A star observed by an 8.2 m telescope in the H band, recorded by a camera with dark current, read noise and
bias. This notebook reduces the frames the way an observer would:

1. take dark frames and science frames;
2. build a master dark, stack the science frames and subtract the dark;
3. measure the flux of the star by aperture photometry and compare its signal-to-noise ratio with the CCD
   equation;
4. run the same steps as **processing elements** in the graph of the context, with a custom **script block**.

Every function propagates the uncertainties of the data (GUM).

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import lumos.io
from lumos.fields import DataProduct
from lumos.processing import photometry, reduction, script
from lumos.processing.photometry import AperturePhotometry, RadialProfile, SignalToNoise

## The observation

The preset `presets/vlt_psf.lumos` holds the star, the telescope (central obstruction and four spiders) and
an ideal camera. We make the star fainter and give the camera realistic defects.

In [ ]:
ctx = lumos.io.load("presets/vlt_psf.lumos")
ctx["star"].flux_density = 2e-4 * u.Jy
camera = ctx["camera"]
camera.dark_current = 200 * u.electron / u.s
camera.read_noise = 8 * u.electron
camera.bias = 500 * u.adu
camera.exposure = 1 * u.s
camera.noise = True  # the preset stores an ideal camera


def take(n):
    frames = []
    for _ in range(n):
        camera.expose()  # a new noise realisation; the clock advances by the exposure time
        frames.append(ctx.run(["camera.out"])["camera.out"])
    return frames


science = take(40)
ctx["star"].flux_density = 0 * u.Jy  # closed shutter: dark frames
darks = take(40)
ctx["star"].flux_density = 2e-4 * u.Jy
print(science[0], "elapsed:", ctx.clock.elapsed)

## Reduction

Without per-frame uncertainties, the stacks estimate them from the scatter of the frames: σ/√N for the mean,
√(π/2) σ/√N for the median. The subtraction adds the uncertainties in quadrature.

In [ ]:
master_dark = reduction.stack(darks, method="median")
stacked = reduction.stack(science, method="sigma_clip", sigma=4)
reduced = reduction.subtract_dark(stacked, master_dark)
reduced = DataProduct(
    reduced.data, axes={k: v.to(u.um) for k, v in reduced.axes.items()}, uncertainty=reduced.uncertainty
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
extent = [
    reduced.axes["x"][0].value,
    reduced.axes["x"][-1].value,
    reduced.axes["y"][0].value,
    reduced.axes["y"][-1].value,
]
axes[0].imshow(science[0].data.value, origin="lower", extent=extent, cmap="gray")
axes[0].set(title="one raw frame [adu]", xlabel="x [µm]", ylabel="y [µm]")
image = axes[1].imshow(
    np.log10(np.maximum(reduced.data.value, 1)), origin="lower", extent=extent, cmap="inferno"
)
axes[1].set(title="reduced stack, log10 [adu]", xlabel="x [µm]")
fig.colorbar(image, ax=axes[1])
fig.tight_layout()

## Photometry and signal-to-noise ratio

The aperture weights every pixel by its exact overlap with the disk; the background annulus removes what is
left of the bias and the dark. The flux is compared with the expected number of electrons (a noiseless run of
the same context), and the scatter of the single-frame measurements with the CCD equation
$S/\sqrt{S + n(1 + n/n_B)(B + D + R^2)}$ (Howell 2006), whose factor $1 + n/n_B$ is the noise of the
background measured over $n_B$ pixels.

In [ ]:
pixel = abs(reduced.axes["x"][1] - reduced.axes["x"][0])
aperture = dict(center=(0 * u.um, 0 * u.um), radius=12 * pixel, annulus=(20 * pixel, 30 * pixel))
result = photometry.aperture_photometry(reduced, **aperture)

camera.noise = False
expected = ctx.run(["camera.out"])["camera.out"]
camera.noise = True
expected = DataProduct(
    expected.data - 500 * u.adu - 200 * u.adu,  # bias and mean dark of 1 s
    axes={k: v.to(u.um) for k, v in expected.axes.items()},
)
truth = photometry.aperture_photometry(expected, center=aperture["center"], radius=aperture["radius"]).flux
print(f"measured flux: {result.flux:.1f} ± {result.uncertainty:.1f}, expected: {truth:.1f}")

single = [
    photometry.aperture_photometry(
        reduction.subtract_dark(
            DataProduct(f.data, axes={k: v.to(u.um) for k, v in f.axes.items()}),
            DataProduct(master_dark.data, axes={k: v.to(u.um) for k, v in f.axes.items()}),
        ),
        **aperture,
    ).flux.value
    for f in science
]
snr_measured = np.mean(single) / np.std(single, ddof=1)
# the median of the annulus raises n/n_B by about π/2
snr_model = photometry.ccd_snr(
    truth.value * u.electron,
    n_pixels=result.area.value,
    n_background=result.background_area.value / (np.pi / 2),
    dark=200 * u.electron,
    read_noise=8 * u.electron,
)
print(f"single-frame SNR: measured {snr_measured:.0f}, CCD equation {snr_model:.0f}")

The measured scatter over 40 frames is itself uncertain by about $1/\sqrt{2(N-1)} \approx 11\,\%$, so the
two SNR values agree within that precision.

## The same pipeline in the graph

Processing blocks are elements with `DataProduct` ports. A **script block** wraps a function of ours, here the
subtraction of the master dark measured above; its parameters (a level in ADU) are saved in `.lumos` files,
its code is not.

In [ ]:
dark_level = float(np.median(master_dark.data.value)) * u.adu


@script(name="examples.pipeline.SubtractLevel", inputs={"in": DataProduct}, outputs={"out": DataProduct})
def subtract_level(frame, *, level: u.Quantity = 0 * u.adu, noise: u.Quantity = 0 * u.adu):
    """Subtract a constant level and attach the uncertainty of a dark-subtracted frame."""
    data = frame.data - level
    sigma = np.sqrt(np.maximum(data.value, 0) + noise.value**2) * u.adu  # photon noise and read noise
    return DataProduct(data, axes={k: v.to(u.um) for k, v in frame.axes.items()}, uncertainty=sigma)


subtract = ctx.add(subtract_level(name="subtract", level=dark_level, noise=np.hypot(8, np.sqrt(200)) * u.adu))
ctx.connect(camera.port("out"), subtract.port("in"))
phot = ctx.add(
    AperturePhotometry(
        name="photometry",
        x=0 * u.um,
        y=0 * u.um,
        radius=aperture["radius"],
        annulus_inner=aperture["annulus"][0],
        annulus_outer=aperture["annulus"][1],
    )
)
profile = ctx.add(RadialProfile(name="profile", x=0 * u.um, y=0 * u.um, width=2 * pixel))
snr = ctx.add(SignalToNoise(name="snr"))
for block in (phot, profile, snr):
    ctx.connect(subtract.port("out"), block.port("in"))

run = ctx.run(["photometry.out", "profile.out", "snr.out"], archive="data-pipeline")
print("flux of one frame:", run["photometry.out"].data[0], "±", run["photometry.out"].uncertainty[0])
print("peak SNR per pixel:", run["snr.out"].data.max())

In [ ]:
radial = run["profile.out"]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(radial.axes["radius"].value, radial.data.value, yerr=radial.uncertainty.value, fmt=".", ms=3)
ax.set(xscale="log", yscale="symlog", xlabel="radius [µm]", ylabel="mean level [adu]", title="Radial profile")
fig.tight_layout()
run.archive.save_figure(
    fig, "radial_profile", data={"radius_um": radial.axes["radius"].value, "profile": radial.data.value}
)

The context, with its processing blocks, is saved like any other one; `lumos.io.load` needs the script
block to be defined first (here, by running the cell above).

In [ ]:
lumos.io.save(ctx, "pipeline.lumos")
lumos.io.load("pipeline.lumos")["subtract"]